# Reconstructing Neutrino Energy from Detector Images

## Goal

In this tutorial, we will train a convolutional neural network (CNN) to estimate the energy of a simulated electron-neutrino interaction from **two detector views**.

By the end, you will be able to:

- explain why neutrino energy matters for oscillation measurements;
- distinguish classification from regression;
- read sparse detector images stored in an HDF5 file;
- build a two-view CNN;
- evaluate average error, bias, resolution, and individual failures.

The scientific question is not only “Does the CNN work?” It is also:

> Where does the CNN improve the estimate, and where does it remain biased?

## 1. The physics story

### Why does energy matter?

Neutrinos can change flavor while traveling. In a simplified two-flavor picture, the oscillation probability contains

$$
P \sim \sin^2(2\theta)\,\sin^2\!\left(1.27\,\frac{\Delta m^2 L}{E}\right).
$$

Here $L$ is the travel distance and $E$ is the neutrino energy. An incorrect reconstructed energy can distort the inferred oscillation pattern.

### What does the detector see?

The incoming neutrino does not leave a visible track before interacting. A simplified electron-neutrino charged-current interaction is

$$
\nu_e + N \rightarrow e^- + X,
$$

where $N$ is a nucleus and $X$ represents the hadronic particles produced in the interaction.

The detector records energy deposited by the outgoing electron and hadrons. Here those deposits are represented as two perpendicular 2D views. The CNN receives both images and predicts one continuous number: the neutrino energy in GeV.

This is **regression**. Classification would instead predict a category such as an interaction type.

### Why is energy reconstruction difficult?

Visible detector activity is not a perfect copy of the original neutrino energy:

- some energy can be carried by particles that are difficult to detect;
- interactions inside the nucleus modify the outgoing particles;
- detector thresholds and nonlinear response affect small deposits;
- showers can overlap or leave the displayed region;
- different interaction modes produce different patterns.

The target called `true_energy` is available because these events are simulated. For real detector data, the true incoming energy is not directly known. A model trained on simulation therefore needs additional validation and uncertainty studies before it could be used in a physics analysis.

## 2. Setup

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

DATA_PATH = Path("data/nue_energy_demo.h5")
MODEL_PATH = Path("models/energy_cnn.keras")
FIGURE_DIR = Path("figures")
MODEL_PATH.parent.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

REFERENCE_COLOR = "#E53935"
CNN_COLOR = "#1546D8"
NEUTRAL_COLOR = "#111827"
GRID_COLOR = "#D1D5DB"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.color": GRID_COLOR,
    "grid.alpha": 0.55,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("TensorFlow:", tf.__version__)
print("Dataset:", DATA_PATH)

## 3. Load and validate the data

In [ ]:
with h5py.File(DATA_PATH, "r") as data_file:
    images = data_file["images"][:]
    true_energy = data_file["true_energy"][:].astype("float32")
    reference_energy = data_file["reco_energy"][:].astype("float32")
    interaction = data_file["interaction"][:]
    split = data_file["split"][:]

expected_length = len(true_energy)
assert images.shape == (expected_length, 2, 100, 80)
assert len(reference_energy) == expected_length
assert len(interaction) == expected_length
assert len(split) == expected_length
assert np.isfinite(true_energy).all()
assert np.isfinite(reference_energy).all()
assert set(np.unique(split)) == {0, 1, 2}

print("Images:", images.shape, images.dtype)
print("True energy:", true_energy.shape, true_energy.dtype)
print("Energy range: {:.3f}–{:.3f} GeV".format(true_energy.min(), true_energy.max()))
print("Nonzero pixel fraction: {:.3%}".format(np.count_nonzero(images) / images.size))

In [ ]:
train_mask = split == 0
validation_mask = split == 1
test_mask = split == 2

x_train, y_train = images[train_mask], true_energy[train_mask]
x_validation, y_validation = images[validation_mask], true_energy[validation_mask]
x_test, y_test = images[test_mask], true_energy[test_mask]
reference_test = reference_energy[test_mask]
interaction_test = interaction[test_mask]

print("Training events:", len(y_train))
print("Validation events:", len(y_validation))
print("Test events:", len(y_test))

The split is stored in the file, so everyone evaluates on the same test events. The test set must remain untouched while the network learns.

In [ ]:
energy_bins = np.arange(0.5, 5.01, 0.25)
figure, axis = plt.subplots(figsize=(8, 4.5))
axis.hist(y_train, bins=energy_bins, density=True, histtype="step", linewidth=2,
          color=REFERENCE_COLOR, label=f"Training (n={len(y_train)})")
axis.hist(y_test, bins=energy_bins, density=True, histtype="step", linewidth=2,
          linestyle="--", color=CNN_COLOR, label=f"Test (n={len(y_test)})")
axis.set(title="True-energy distribution", xlabel="True neutrino energy (GeV)",
         ylabel="Normalized event density")
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / "energy_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Look at one simulated event

In [ ]:
median_test_energy = np.median(y_test)
display_index = int(np.argmin(np.abs(y_test - median_test_energy)))

figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for view_index, axis in enumerate(axes):
    image = axis.imshow(x_test[display_index, view_index], origin="lower",
                        aspect="auto", cmap="magma", vmin=0, vmax=80)
    axis.set_title(f"Detector view {view_index}")
    axis.set_xlabel("Detector column")
    axis.set_ylabel("Detector row")
    axis.grid(False)

figure.suptitle(f"Example event: true energy = {y_test[display_index]:.3f} GeV")
figure.colorbar(image, ax=axes, label="Stored pixel intensity", shrink=0.85)
plt.show()

Most pixels are black because a neutrino interaction activates only a small fraction of the detector cells. Bright and extended patterns contain information about the particles and their energies.

## 5. Build a two-view CNN

Each detector view is processed by its own CNN branch. Convolutional layers search for local patterns such as tracks and showers. The two learned representations are then combined to predict one energy value.

The stored pixel intensities range from 0 to 255, so the first layer divides them by 255. This makes the input scale explicit and easy to interpret.

In [ ]:
def build_view_branch(name):
    view_input = keras.Input(shape=(100, 80, 1), name=f"{name}_input")
    x = layers.Rescaling(1.0 / 255.0, name=f"{name}_normalization")(view_input)

    for filters in (16, 32, 64):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    return view_input, x


view_0_input, view_0_features = build_view_branch("view_0")
view_1_input, view_1_features = build_view_branch("view_1")

combined_features = layers.Concatenate()([view_0_features, view_1_features])
combined_features = layers.Dense(64, activation="relu")(combined_features)
energy_output = layers.Dense(1, name="energy")(combined_features)

model = keras.Model(
    inputs=[view_0_input, view_1_input],
    outputs=energy_output,
    name="two_view_energy_cnn",
)

optimizer = keras.optimizers.legacy.Adam(learning_rate=5e-4)
model.compile(optimizer=optimizer, loss="mae", metrics=["mae"])
model.summary()

The loss is **mean absolute error**:

$$
\mathrm{MAE}=\frac{1}{N}\sum_i\left|E_i^{\mathrm{pred}}-E_i^{\mathrm{true}}\right|.
$$

MAE is measured directly in GeV and is less dominated by rare large errors than mean squared error.

## 6. Train the model

In [ ]:
def model_inputs(image_array):
    return [
        image_array[:, 0, :, :, None],
        image_array[:, 1, :, :, None],
    ]


callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
    ),
]

history = model.fit(
    model_inputs(x_train),
    y_train,
    validation_data=(model_inputs(x_validation), y_validation),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=2,
)

model = keras.models.load_model(MODEL_PATH)
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
print("Epochs completed:", len(history.history["loss"]))
print("Best validation epoch:", best_epoch)

In [ ]:
figure, axis = plt.subplots(figsize=(10, 6))
epochs = np.arange(1, len(history.history["mae"]) + 1)
axis.plot(epochs, history.history["mae"], marker="o", markersize=4,
          color="#6B7280", linewidth=1.8, alpha=0.8, label="Training loss")
axis.plot(epochs, history.history["val_mae"], marker="o", markersize=5,
          color=CNN_COLOR, linewidth=2.2, label="Validation loss")
axis.axvline(best_epoch, color=NEUTRAL_COLOR, linestyle="--", linewidth=1.5,
             label=f"Best validation epoch: {best_epoch}")
axis.set(title="NuE Energy CNN: Loss over epochs",
         xlabel="Epoch", ylabel="Mean absolute error (GeV)")
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / "validation_loss_over_epochs.png", dpi=160,
               bbox_inches="tight")
plt.show()

## 7. Evaluate on unseen test events

In [ ]:
predicted_test = model.predict(model_inputs(x_test), batch_size=64).reshape(-1)
constant_test = np.full_like(y_test, np.median(y_train))


def mean_absolute_error(prediction, truth=y_test):
    return float(np.mean(np.abs(prediction - truth)))


def root_mean_squared_error(prediction, truth=y_test):
    return float(np.sqrt(np.mean((prediction - truth) ** 2)))


predictions = {
    "Constant baseline": constant_test,
    "Reference reconstruction": reference_test,
    "CNN": predicted_test,
}

print(f"{'Method':26s} {'MAE (GeV)':>11s} {'RMSE (GeV)':>12s}")
print("-" * 52)
for method, prediction in predictions.items():
    print(f"{method:26s} {mean_absolute_error(prediction):11.3f} "
          f"{root_mean_squared_error(prediction):12.3f}")

The constant baseline always predicts the median training energy. Beating it shows that the images contain useful information. The reference reconstruction is a stronger physics baseline included in the dataset.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
plot_min, plot_max = 0.5, 5.0

for axis, prediction, title in [
    (axes[0], reference_test, "Reference reconstruction"),
    (axes[1], predicted_test, "CNN reconstruction"),
]:
    axis.scatter(y_test, prediction, s=18, alpha=0.45, color=REFERENCE_COLOR,
                 edgecolors="none")
    axis.plot([plot_min, plot_max], [plot_min, plot_max], color=NEUTRAL_COLOR,
              linestyle="--", label="Perfect prediction")
    outside = int(((prediction < plot_min) | (prediction > plot_max)).sum())
    axis.text(0.04, 0.96,
              f"MAE = {mean_absolute_error(prediction):.3f} GeV\n"
              f"RMSE = {root_mean_squared_error(prediction):.3f} GeV\n"
              f"Outside plot range = {outside}",
              transform=axis.transAxes, va="top",
              bbox={"facecolor": "white", "edgecolor": NEUTRAL_COLOR, "alpha": 0.9})
    axis.set(title=title, xlabel="True neutrino energy (GeV)",
             xlim=(plot_min, plot_max), ylim=(plot_min, plot_max))
    axis.set_aspect("equal", adjustable="box")
    axis.legend(loc="lower right")

axes[0].set_ylabel("Reconstructed neutrino energy (GeV)")
figure.suptitle("Energy reconstruction on the same test events")
figure.tight_layout()
figure.savefig(FIGURE_DIR / "predicted_vs_true.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Fractional error and energy-dependent bias

The fractional error compares the error with the event's true energy:

$$
\delta = \frac{E_{\mathrm{reco}}-E_{\mathrm{true}}}{E_{\mathrm{true}}}.
$$

- $\delta=0$: perfect reconstruction
- $\delta>0$: energy is overestimated
- $\delta<0$: energy is underestimated

In [ ]:
reference_fractional_error = (reference_test - y_test) / y_test
cnn_fractional_error = (predicted_test - y_test) / y_test


def central_68_half_width(values):
    lower, upper = np.percentile(values, [16, 84])
    return float((upper - lower) / 2.0)


def binned_fractional_statistics(truth, fractional_error, edges):
    centers, means, spreads, mean_errors, resolutions, counts = [], [], [], [], [], []
    for index, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        if index == len(edges) - 2:
            mask = (truth >= low) & (truth <= high)
        else:
            mask = (truth >= low) & (truth < high)

        values = fractional_error[mask]
        centers.append((low + high) / 2.0)
        means.append(values.mean())
        spreads.append(values.std(ddof=1))
        mean_errors.append(values.std(ddof=1) / np.sqrt(len(values)))
        resolutions.append(central_68_half_width(values))
        counts.append(len(values))

    return tuple(np.asarray(item) for item in
                 (centers, means, spreads, mean_errors, resolutions, counts))


bin_edges = np.arange(0.5, 5.01, 0.5)
reference_stats = binned_fractional_statistics(
    y_test, reference_fractional_error, bin_edges
)
cnn_stats = binned_fractional_statistics(y_test, cnn_fractional_error, bin_edges)

print("Central 68% fractional half-width")
print(f"  Reference: {central_68_half_width(reference_fractional_error):.3f}")
print(f"  CNN:       {central_68_half_width(cnn_fractional_error):.3f}")

In [ ]:
figure, axis = plt.subplots(figsize=(10, 6))
fractional_bins = np.linspace(-1, 1, 81)
x_values = np.linspace(-1, 1, 600)

for values, label, color, style in [
    (reference_fractional_error, "Reference", REFERENCE_COLOR, "-"),
    (cnn_fractional_error, "CNN", CNN_COLOR, "-"),
]:
    mean = float(np.mean(values))
    standard_deviation = float(np.std(values, ddof=1))
    gaussian = np.exp(-0.5 * ((x_values - mean) / standard_deviation) ** 2)
    gaussian /= standard_deviation * np.sqrt(2 * np.pi)

    axis.hist(values, bins=fractional_bins, density=True, histtype="step",
              linewidth=2.2, linestyle=style, color=color, label=f"{label} events")
    axis.plot(x_values, gaussian, color=color, linestyle="--", linewidth=2,
              label=f"{label} Gaussian guide")
    axis.text(
        0.67,
        0.88 if label == "Reference" else 0.72,
        f"{label}\nMean = {mean:+.3f}\nStd. dev. = {standard_deviation:.3f}",
        transform=axis.transAxes,
        color=color,
        fontsize=11,
        bbox={"facecolor": "white", "edgecolor": color, "alpha": 0.85},
    )

axis.axvline(0, color=NEUTRAL_COLOR, linestyle=":", linewidth=1.5)
axis.set(title="NuE Energy Reconstruction",
         xlabel=r"$(E_{reco}-E_{true})/E_{true}$",
         ylabel="Normalized event density", xlim=(-1, 1))
axis.legend(loc="upper left", fontsize=9)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "nue_energy_fractional_error.png", dpi=160,
               bbox_inches="tight")
plt.show()


figure, axis = plt.subplots(figsize=(10, 6))

for stats, label, color, marker, style in [
    (reference_stats, "Reference", REFERENCE_COLOR, "o", "-"),
    (cnn_stats, "CNN", CNN_COLOR, "s", "-"),
]:
    centers, means, spreads, _, _, _ = stats
    offset = -0.025 if label == "Reference" else 0.025
    axis.errorbar(centers + offset, means, yerr=spreads, marker=marker,
                  linestyle=style, linewidth=2, capsize=3, alpha=0.8,
                  color=color, label=f"{label}: mean ± 1σ spread")

axis.axhline(0, color=NEUTRAL_COLOR, linestyle="--", linewidth=1.7)
axis.set(title="Prediction Bias – NuE Energy",
         xlabel=r"$E_{true}$ (GeV)",
         ylabel=r"Mean $(E_{reco}-E_{true})/E_{true}$")
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / "nue_energy_prediction_bias.png", dpi=160,
               bbox_inches="tight")
plt.show()

In [ ]:
print(f"{'Energy bin':15s} {'Events':>7s} {'Reference bias':>17s} {'CNN bias':>11s}")
print("-" * 55)
for index, (low, high) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
    count = int(cnn_stats[5][index])
    print(f"{low:.1f}–{high:.1f} GeV {count:7d} "
          f"{reference_stats[1][index]:+17.3f} {cnn_stats[1][index]:+11.3f}")

The end bins contain few events, so their means are uncertain. However, the pattern also appears in bins with about 60 events: the CNN tends to overestimate below the dense central region and underestimate above it. A better global MAE does not guarantee zero bias in every energy range.

## 9. Energy resolution

Here, resolution is summarized by half the distance between the 16th and 84th percentiles of the fractional-error distribution in each energy bin. A smaller value means a narrower central error distribution.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5))

for stats, label, color, marker, style in [
    (reference_stats, "Reference", REFERENCE_COLOR, "o", "-"),
    (cnn_stats, "CNN", CNN_COLOR, "s", "-"),
]:
    centers, _, _, _, resolutions, _ = stats
    axis.plot(centers, resolutions, marker=marker, linestyle=style,
              linewidth=2, color=color, label=label)

axis.set(title="NuE Energy Resolution",
         xlabel=r"$E_{true}$ (GeV)",
         ylabel="Central 68% fractional half-width", ylim=(0, None))
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / "nue_energy_resolution.png", dpi=160,
               bbox_inches="tight")
plt.show()

print("Events per bin:", cnn_stats[5].astype(int).tolist())

## 10. Inspect individual events

In [ ]:
cnn_absolute_error = np.abs(predicted_test - y_test)
reference_absolute_error = np.abs(reference_test - y_test)

typical_index = int(np.argmin(np.abs(cnn_absolute_error - np.median(cnn_absolute_error))))
cnn_improves_index = int(np.argmax(reference_absolute_error - cnn_absolute_error))
reference_improves_index = int(np.argmax(cnn_absolute_error - reference_absolute_error))

selected_events = [
    (typical_index, "Typical test event"),
    (cnn_improves_index, "CNN improves most"),
    (reference_improves_index, "Reference improves most"),
]

interaction_names = {4: "QE", 5: "RES", 6: "DIS", 7: "Other"}

figure, axes = plt.subplots(3, 2, figsize=(12, 15), constrained_layout=True)
for row, (event_index, description) in enumerate(selected_events):
    for view_index in range(2):
        axis = axes[row, view_index]
        axis.imshow(x_test[event_index, view_index], origin="lower",
                    aspect="auto", cmap="magma", vmin=0, vmax=80)
        axis.set_title(f"{description}: view {view_index}")
        axis.set_xlabel("Detector column")
        axis.set_ylabel("Detector row")
        axis.grid(False)

        if view_index == 0:
            mode = interaction_names.get(int(interaction_test[event_index]), "Unknown")
            axis.text(
                0.02, 0.98,
                f"Mode: {mode}\n"
                f"True: {y_test[event_index]:.3f} GeV\n"
                f"CNN: {predicted_test[event_index]:.3f} GeV\n"
                f"Reference: {reference_test[event_index]:.3f} GeV",
                transform=axis.transAxes,
                va="top",
                color="white",
                bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.7},
            )

figure.suptitle("Example test events", fontsize=16)
figure.savefig(FIGURE_DIR / "event_examples.png", dpi=160, bbox_inches="tight")
plt.show()

## Checks and takeaways

- The CNN beats the constant baseline, so it learns information from the detector images.
- On this fixed test split, it also reduces overall MAE, RMSE, and the central fractional-error width relative to the reference reconstruction.
- It does **not** improve every event or every energy range.
- The training spectrum is concentrated near 2 GeV, and the CNN tends to pull rare energies toward that region.
- Looking at distributions, energy bins, and individual failures gives a more honest picture than quoting one score.

## Next steps

Try one change at a time:

1. remove one detector view and measure the performance change;
2. balance the training events across energy bins;
3. compare MAE with a relative-error loss;
4. separate the results by interaction mode;
5. inspect the largest CNN failures and propose a physical explanation.

Remember: this compact simulation tutorial demonstrates a method. A real physics analysis would also require detector calibration, systematic uncertainties, and validation against real data.